# 🧩 Mini-Lab: Reproducibility Audit

**Module: Escape the Notebook: Environment, Repo, Git** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Explain** what makes a repository reproducible from a clean clone
2. **Identify** the essential files and practices that enable reproducibility
3. **Compare** a reproducible repo against a non-reproducible one using a concrete checklist

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Reproducibility | Someone else's machine reproduces your result from documented, pinned steps, starting from a fresh repo clone with no hidden setup. |


## The Clone-and-Run Test

Imagine a colleague just joined your team. They do:

```bash
git clone <your-repo-url>
cd your-project
# ... now what?
```

**Reproducibility** means that a stranger — with no Slack messages, no "ask Sarah" shortcuts — can clone, install, and run your project by following what's *in the repo itself*.

If they can't, the project is **not reproducible**. Let's see what that looks like in practice.

## Two Repos Side by Side

Below we simulate the directory listings of two repos. One is reproducible; the other is not. We'll inspect both and score them against a checklist.

In [1]:
# Simulated directory listings for two repos

repo_a = {
    "name": "Repo A",
    "files": [
        "app.py",
        "utils.py",
        "notebook.ipynb",
        "data.csv",
    ],
    "has_readme": False,
    "has_requirements": False,
    "has_gitignore": False,
    "has_env_example": False,
    "has_pinned_versions": False,
    "secrets_in_code": True,
}

repo_b = {
    "name": "Repo B",
    "files": [
        "README.md",
        "requirements.txt",
        ".gitignore",
        ".env.example",
        "src/app.py",
        "src/utils.py",
        "tests/test_app.py",
        "data/sample.csv",
    ],
    "has_readme": True,
    "has_requirements": True,
    "has_gitignore": True,
    "has_env_example": True,
    "has_pinned_versions": True,
    "secrets_in_code": False,
}

## The Reproducibility Checklist

We'll score each repo against the key questions a newcomer asks when they clone a project.

In [2]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


# Define the reproducibility checklist
checklist = [
    ("README with setup instructions",   "has_readme",
     "Without a README, the newcomer has no idea how to install or run the project."),
    ("Dependency file (requirements.txt)", "has_requirements",
     "Without this, they must guess which packages to install — and which versions."),
    ("Pinned dependency versions",         "has_pinned_versions",
     "Unpinned versions mean `pip install` grabs the latest, which may break the code."),
    (".gitignore present",                 "has_gitignore",
     "Without .gitignore, junk files (caches, .env, OS files) pollute the repo."),
    (".env.example for secrets",           "has_env_example",
     "Shows what env vars are needed without leaking actual values."),
    ("No secrets hard-coded in source",    "secrets_in_code",
     "Hard-coded API keys get committed to Git history — a security disaster."),
]

In [3]:
def audit_repo(repo):
    """Score a repo against the reproducibility checklist."""
    lines = [f"### Audit: {repo['name']}\n"]
    lines.append(f"**Files:** `{'`, `'.join(repo['files'])}`\n")
    lines.append("| # | Check | Pass? | Why It Matters |")
    lines.append("|---|-------|-------|----------------|")

    score = 0
    for i, (label, key, reason) in enumerate(checklist, 1):
        value = repo[key]
        # For secrets_in_code, passing means secrets are NOT in code
        if key == "secrets_in_code":
            passed = not value
        else:
            passed = value
        icon = "✅" if passed else "❌"
        score += int(passed)
        lines.append(f"| {i} | {label} | {icon} | {reason} |")

    lines.append(f"\n**Score: {score}/{len(checklist)}**\n")
    return "\n".join(lines)


md(audit_repo(repo_a))

### Audit: Repo A

**Files:** `app.py`, `utils.py`, `notebook.ipynb`, `data.csv`

| # | Check | Pass? | Why It Matters |
|---|-------|-------|----------------|
| 1 | README with setup instructions | ❌ | Without a README, the newcomer has no idea how to install or run the project. |
| 2 | Dependency file (requirements.txt) | ❌ | Without this, they must guess which packages to install — and which versions. |
| 3 | Pinned dependency versions | ❌ | Unpinned versions mean `pip install` grabs the latest, which may break the code. |
| 4 | .gitignore present | ❌ | Without .gitignore, junk files (caches, .env, OS files) pollute the repo. |
| 5 | .env.example for secrets | ❌ | Shows what env vars are needed without leaking actual values. |
| 6 | No secrets hard-coded in source | ❌ | Hard-coded API keys get committed to Git history — a security disaster. |

**Score: 0/6**


In [4]:
md(audit_repo(repo_b))

### Audit: Repo B

**Files:** `README.md`, `requirements.txt`, `.gitignore`, `.env.example`, `src/app.py`, `src/utils.py`, `tests/test_app.py`, `data/sample.csv`

| # | Check | Pass? | Why It Matters |
|---|-------|-------|----------------|
| 1 | README with setup instructions | ✅ | Without a README, the newcomer has no idea how to install or run the project. |
| 2 | Dependency file (requirements.txt) | ✅ | Without this, they must guess which packages to install — and which versions. |
| 3 | Pinned dependency versions | ✅ | Unpinned versions mean `pip install` grabs the latest, which may break the code. |
| 4 | .gitignore present | ✅ | Without .gitignore, junk files (caches, .env, OS files) pollute the repo. |
| 5 | .env.example for secrets | ✅ | Shows what env vars are needed without leaking actual values. |
| 6 | No secrets hard-coded in source | ✅ | Hard-coded API keys get committed to Git history — a security disaster. |

**Score: 6/6**


## What Each Piece Does

Let's look at the role of each file in Repo B — the reproducible one — and map it to the newcomer's question it answers.

In [5]:
file_roles = [
    ("README.md",         "How do I set up and run this?",
     "Step-by-step instructions: install, configure, run."),
    ("requirements.txt",  "What packages do I need?",
     "Exact list of dependencies with pinned versions."),
    (".gitignore",        "What should NOT be in the repo?",
     "Keeps .env, __pycache__, .DS_Store, etc. out of version control."),
    (".env.example",      "What secrets/config do I need to provide?",
     "Template showing required env vars (OPENAI_API_KEY=your-key-here)."),
    ("src/",              "Where is the actual code?",
     "Organized source code, not scattered loose files."),
    ("tests/",            "How do I know it works?",
     "Runnable tests that verify the code still behaves correctly."),
]

rows = ["| File / Folder | Newcomer's Question | What It Provides |"]
rows.append("|---------------|---------------------|------------------|")
for f, q, a in file_roles:
    rows.append(f"| `{f}` | {q} | {a} |")

md("\n".join(rows))

| File / Folder | Newcomer's Question | What It Provides |
|---------------|---------------------|------------------|
| `README.md` | How do I set up and run this? | Step-by-step instructions: install, configure, run. |
| `requirements.txt` | What packages do I need? | Exact list of dependencies with pinned versions. |
| `.gitignore` | What should NOT be in the repo? | Keeps .env, __pycache__, .DS_Store, etc. out of version control. |
| `.env.example` | What secrets/config do I need to provide? | Template showing required env vars (OPENAI_API_KEY=your-key-here). |
| `src/` | Where is the actual code? | Organized source code, not scattered loose files. |
| `tests/` | How do I know it works? | Runnable tests that verify the code still behaves correctly. |

## The Golden Rule

> **A project is reproducible when a stranger can clone it and run it using only what's in the repo.**

No Slack messages. No "works on my machine." No guessing package versions.

Every item on the checklist removes one source of guesswork:

- **README** → removes "how do I start?" guesswork
- **Pinned requirements** → removes version guesswork
- **.env.example** → removes "what secrets do I need?" guesswork
- **.gitignore** → removes junk that obscures the real code
- **No hard-coded secrets** → removes security risk

In the upcoming mini-labs, we'll create each of these pieces one by one.

## 🎯 Summary

### Key Takeaways

1. **Reproducibility** — a project is reproducible when a stranger can clone it and run it using only what's in the repo, with zero outside knowledge
2. **The checklist** — README, pinned requirements, .gitignore, .env.example, and no hard-coded secrets are the essential ingredients
3. **Each file answers a question** — every reproducibility artifact exists to remove one specific source of guesswork for the newcomer